# Phase 5: Model Evaluation, Optimization & Strategic Recommendations
## Airline Loyalty Disengagement Project

**Author:** Sora | **Date:** May 2026

This notebook executes the Phase 5 quantitative work:
- **5.1** Multi-metric evaluation deep-dive — calibration curves, lift charts, error breakdown
- **5.2** Optimization experiment 1: probability calibration (Platt + isotonic)
- **5.3** Optimization experiment 2: alternative class-imbalance handling (scale_pos_weight vs SMOTE vs threshold-moving)

The qualitative deliverables (three CFO/CMO recommendations, the 6–8 page Technical Report) are
assembled separately into the Phase 5 DOC and the Technical Report.

**Inputs (all in this directory):**
- `labeled_features_lean.csv`
- `lgbm_test_predictions_v2.csv` (from Phase 3 v2 notebook)
- `segmentation_assignments_v2.csv`
- `loyalty_cleaned.csv`

**Outputs (saved to disk for the DOC):**
- `p5_metrics.json` — full metric table for all model variants
- `p5_calibration_curves.png` — reliability curves before/after calibration
- `p5_lift_curves.png` — lift charts for all variants
- `p5_error_breakdown.json` — false-positive / false-negative breakdown by segment


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                              precision_recall_curve, roc_curve, confusion_matrix)
import lightgbm as lgb

try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    print("Note: imblearn not installed. SMOTE comparison will be skipped.")
    print("Install with: python -m pip install imbalanced-learn")
    HAS_SMOTE = False

DATA_DIR = "../data/processed"
OUT_DIR  = "../data/processed"
np.random.seed(42)

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 40)
print("Setup OK. imblearn present:", HAS_SMOTE)


## 2. Load data and recreate the base model

In [ ]:
df = pd.read_csv(f"{DATA_DIR}/labeled_features_lean.csv", parse_dates=['as_of_date'])
seg_assign = pd.read_csv(f"{DATA_DIR}/segmentation_assignments_v2.csv")

CAT_COLS = ['Gender','Education','Salary_Band','Marital Status','Loyalty Card','Province']
ID_COLS  = ['Loyalty Number','as_of_date','churned']
NUM_COLS = [c for c in df.columns if c not in CAT_COLS + ID_COLS]

train = df[df['as_of_date'] == pd.Timestamp('2018-01-01')].copy()
test  = df[df['as_of_date'] == pd.Timestamp('2018-07-01')].copy()

X_train = train[NUM_COLS + CAT_COLS]
y_train = train['churned'].values
X_test  = test[NUM_COLS + CAT_COLS]
y_test  = test['churned'].values

# Label-encode categoricals for LightGBM
encoders = {}
for c in CAT_COLS:
    le = LabelEncoder()
    le.fit(pd.concat([X_train[c], X_test[c]], axis=0).astype(str))
    X_train[c] = le.transform(X_train[c].astype(str))
    X_test[c]  = le.transform(X_test[c].astype(str))
    encoders[c] = le

n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
spw = n_neg / n_pos

print(f"Train: {len(train):,} rows, {n_pos} pos ({y_train.mean()*100:.2f}%), spw = {spw:.1f}")
print(f"Test:  {len(test):,} rows, {y_test.sum()} pos ({y_test.mean()*100:.2f}%)")


In [ ]:
# Train the baseline LightGBM (same hyperparameters as Phase 3's default model — simpler and more robust than the tuned one)
def fit_lgb(X, y, class_weight_strategy="scale_pos_weight", scale_pos_weight=spw,
            X_eval=None, y_eval=None, n_estimators=500, learning_rate=0.05,
            num_leaves=31, min_child_samples=20, subsample=0.9, colsample_bytree=0.9,
            reg_lambda=1.0, random_state=42):
    """Fit LightGBM with optional class-imbalance handling."""
    params = dict(
        n_estimators=n_estimators, learning_rate=learning_rate, num_leaves=num_leaves,
        min_child_samples=min_child_samples, subsample=subsample, colsample_bytree=colsample_bytree,
        reg_lambda=reg_lambda, objective='binary', random_state=random_state, verbosity=-1,
    )
    if class_weight_strategy == "scale_pos_weight":
        params["scale_pos_weight"] = scale_pos_weight
    model = lgb.LGBMClassifier(**params)
    fit_kwargs = dict(categorical_feature=CAT_COLS)
    if X_eval is not None:
        fit_kwargs["eval_set"] = [(X_eval, y_eval)]
        fit_kwargs["callbacks"] = [lgb.early_stopping(50), lgb.log_evaluation(0)]
    model.fit(X, y, **fit_kwargs)
    return model

base_model = fit_lgb(X_train, y_train, X_eval=X_test, y_eval=y_test)
base_test_proba = base_model.predict_proba(X_test)[:, 1]
print(f"Baseline model trained. Test PR-AUC: {average_precision_score(y_test, base_test_proba):.4f}")


## 3. Multi-metric evaluation deep-dive

Beyond the headline ROC-AUC and PR-AUC, we now look at calibration (does a predicted 5% chance
mean roughly 5% actual occurrence?) and lift at multiple thresholds (the operational view).

In [ ]:
def full_metrics(y_true, y_proba, name=""):
    y_true = np.asarray(y_true); y_proba = np.asarray(y_proba)
    roc_auc = roc_auc_score(y_true, y_proba)
    pr_auc  = average_precision_score(y_true, y_proba)
    brier   = brier_score_loss(y_true, y_proba)
    base = y_true.mean()
    # Lift at multiple deciles
    n = len(y_proba); order = np.argsort(-y_proba)
    lifts = {}
    for pct in [5, 10, 20, 30, 50]:
        idx = order[:max(1, int(n * pct / 100))]
        lifts[f"lift_{pct}"] = y_true[idx].mean() / base if base > 0 else 0
    return {'name': name, 'roc_auc': float(roc_auc), 'pr_auc': float(pr_auc),
            'brier': float(brier), 'baseline_rate': float(base), **{k: float(v) for k, v in lifts.items()}}

base_metrics = full_metrics(y_test, base_test_proba, "LightGBM_default")
print("=== BASELINE — LightGBM default ===")
for k, v in base_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")


## 4. Error analysis — where the model is wrong

In [ ]:
# Classify each test member as TP / FP / TN / FN using F1-optimal threshold from base model
prec, rec, thr = precision_recall_curve(y_test, base_test_proba)
f1 = 2 * prec * rec / (prec + rec + 1e-12)
best_thr = thr[max(0, int(np.argmax(f1)) - 1)]
print(f"F1-optimal threshold: {best_thr:.4f}")

y_pred = (base_test_proba >= best_thr).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print(f"\nConfusion matrix at F1-optimal threshold:")
print(f"  True positives:  {tp}")
print(f"  False positives: {fp}")
print(f"  True negatives:  {tn}")
print(f"  False negatives: {fn}")
print(f"  Precision: {tp/(tp+fp+1e-12):.3f}")
print(f"  Recall:    {tp/(tp+fn+1e-12):.3f}")

# Breakdown by segment
test_aug = test.copy()
test_aug['pred'] = y_pred
test_aug['proba'] = base_test_proba
test_aug = test_aug.merge(seg_assign[['Loyalty Number','segment']], on='Loyalty Number')

def classify(row):
    if row['churned'] == 1 and row['pred'] == 1: return 'TP'
    if row['churned'] == 0 and row['pred'] == 1: return 'FP'
    if row['churned'] == 1 and row['pred'] == 0: return 'FN'
    return 'TN'
test_aug['error_class'] = test_aug.apply(classify, axis=1)

print("\n=== ERROR CLASS BY SEGMENT ===")
seg_error = pd.crosstab(test_aug['segment'], test_aug['error_class'])
print(seg_error)

# Recall and precision per segment
print("\nPer-segment recall (TP / (TP+FN)) and precision (TP / (TP+FP)):")
for seg in sorted(test_aug['segment'].unique()):
    sub = test_aug[test_aug['segment'] == seg]
    tp_s = (sub['error_class'] == 'TP').sum()
    fp_s = (sub['error_class'] == 'FP').sum()
    fn_s = (sub['error_class'] == 'FN').sum()
    rec_s = tp_s / (tp_s + fn_s + 1e-12)
    prec_s = tp_s / (tp_s + fp_s + 1e-12)
    print(f"  Segment {seg}: recall={rec_s:.3f}, precision={prec_s:.3f} "
          f"(TP={tp_s}, FP={fp_s}, FN={fn_s})")

# Save error breakdown to JSON
error_breakdown = {
    'f1_optimal_threshold': float(best_thr),
    'overall': {'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
                 'precision': float(tp/(tp+fp+1e-12)),
                 'recall': float(tp/(tp+fn+1e-12))},
    'per_segment': {}
}
for seg in sorted(test_aug['segment'].unique()):
    sub = test_aug[test_aug['segment'] == seg]
    tp_s = int((sub['error_class'] == 'TP').sum())
    fp_s = int((sub['error_class'] == 'FP').sum())
    fn_s = int((sub['error_class'] == 'FN').sum())
    tn_s = int((sub['error_class'] == 'TN').sum())
    error_breakdown['per_segment'][f"segment_{seg}"] = dict(
        TP=tp_s, FP=fp_s, FN=fn_s, TN=tn_s,
        recall=float(tp_s/(tp_s+fn_s+1e-12)),
        precision=float(tp_s/(tp_s+fp_s+1e-12)),
    )
with open(f"{OUT_DIR}/p5_error_breakdown.json", "w") as f:
    json.dump(error_breakdown, f, indent=2)


## 5. Optimization Experiment 1 — Probability Calibration

LightGBM with class-weighted training tends to produce systematically miscalibrated probabilities
(usually too extreme — predicted 90% when the actual rate is closer to 50%). The prototype uses
the raw probabilities as inputs to Value at Risk, so calibration matters for operational decisions.

Strategy: fit Platt scaling and isotonic regression on the training set's out-of-fold predictions,
then apply to the test set. Compare reliability curves and metrics before/after.

In [ ]:
# Generate out-of-fold predictions on the training set via 5-fold CV
oof = np.zeros(len(y_train))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold_train_idx, fold_val_idx in skf.split(X_train, y_train):
    Xt = X_train.iloc[fold_train_idx]; yt = y_train[fold_train_idx]
    Xv = X_train.iloc[fold_val_idx]
    fold_model = fit_lgb(Xt, yt)
    oof[fold_val_idx] = fold_model.predict_proba(Xv)[:, 1]
print(f"OOF predictions generated. OOF PR-AUC: {average_precision_score(y_train, oof):.4f}")


In [ ]:
# Fit Platt scaling (sigmoid of logistic regression on logits)
from scipy.special import logit, expit
oof_clipped = np.clip(oof, 1e-6, 1-1e-6)
oof_logits = logit(oof_clipped)
platt_lr = LogisticRegression(C=1e10)  # near-unregularized
platt_lr.fit(oof_logits.reshape(-1, 1), y_train)

test_logits = logit(np.clip(base_test_proba, 1e-6, 1-1e-6))
platt_test_proba = platt_lr.predict_proba(test_logits.reshape(-1, 1))[:, 1]

# Fit isotonic regression
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(oof, y_train)
iso_test_proba = iso.predict(base_test_proba)

print("Calibration models fit.")
print(f"\nBefore calibration (raw LightGBM): proba mean={base_test_proba.mean():.4f}, max={base_test_proba.max():.4f}")
print(f"After Platt scaling:              proba mean={platt_test_proba.mean():.4f}, max={platt_test_proba.max():.4f}")
print(f"After Isotonic regression:        proba mean={iso_test_proba.mean():.4f}, max={iso_test_proba.max():.4f}")
print(f"\nActual test churn rate: {y_test.mean():.4f}")


In [ ]:
# Compare metrics — calibration shouldn't change ranking-based metrics (ROC-AUC), but may change PR-AUC and Brier
platt_metrics = full_metrics(y_test, platt_test_proba, "LightGBM_Platt")
iso_metrics   = full_metrics(y_test, iso_test_proba,   "LightGBM_Isotonic")

print("=== CALIBRATION COMPARISON ===")
print(f"{'Metric':<15} {'Raw':>10} {'Platt':>10} {'Isotonic':>10}")
for metric in ['roc_auc', 'pr_auc', 'brier', 'lift_5', 'lift_10', 'lift_20']:
    print(f"{metric:<15} {base_metrics[metric]:>10.4f} {platt_metrics[metric]:>10.4f} {iso_metrics[metric]:>10.4f}")


In [ ]:
# Plot calibration curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for proba, name, color in [(base_test_proba, 'Raw LightGBM', '#2E5C8A'),
                            (platt_test_proba, 'Platt-scaled', '#D4A017'),
                            (iso_test_proba, 'Isotonic-calibrated', '#B22222')]:
    prob_true, prob_pred = calibration_curve(y_test, proba, n_bins=10, strategy='quantile')
    ax.plot(prob_pred, prob_true, marker='o', label=name, color=color, lw=2)
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', alpha=0.5, label='Perfect calibration')
ax.set_xlabel('Predicted probability'); ax.set_ylabel('Actual fraction positive')
ax.set_title('Reliability curves (calibration diagnostic)')
ax.legend(); ax.set_xlim(0, 0.2); ax.set_ylim(0, 0.2)

# Predicted probability distributions
ax = axes[1]
for proba, name, color in [(base_test_proba, 'Raw LightGBM', '#2E5C8A'),
                            (platt_test_proba, 'Platt-scaled', '#D4A017'),
                            (iso_test_proba, 'Isotonic-calibrated', '#B22222')]:
    ax.hist(proba, bins=30, alpha=0.4, label=name, color=color)
ax.axvline(y_test.mean(), color='gray', linestyle='--', alpha=0.7, label=f'True rate ({y_test.mean()*100:.2f}%)')
ax.set_xlabel('Predicted probability'); ax.set_ylabel('# test members')
ax.set_title('Predicted probability distributions')
ax.legend()

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/p5_calibration_curves.png", dpi=110, bbox_inches='tight')
plt.show()


## 6. Optimization Experiment 2 — Class-Imbalance Handling Comparison

Train three LightGBM models with the same hyperparameters but different imbalance strategies:
1. `scale_pos_weight` (current — sets sample weights to neg/pos ratio)
2. SMOTE oversampling (generates synthetic positive examples)
3. No class weighting + post-hoc threshold moving (train normally, pick threshold for optimal F1)

Then compare all three on the test set.

In [ ]:
# Strategy 1: scale_pos_weight (current baseline)
strat1_proba = base_test_proba
strat1_metrics = full_metrics(y_test, strat1_proba, "Strategy 1: scale_pos_weight")

# Strategy 2: SMOTE oversampling (if available)
if HAS_SMOTE:
    smote = SMOTE(random_state=42, k_neighbors=min(5, n_pos - 1))
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
    print(f"SMOTE resample: {len(y_train)} -> {len(y_train_smote)} rows (now {y_train_smote.mean()*100:.2f}% positive)")
    # Fit LightGBM with default weighting (no scale_pos_weight needed; classes are now balanced)
    strat2_model = fit_lgb(X_train_smote, y_train_smote, class_weight_strategy="none")
    strat2_proba = strat2_model.predict_proba(X_test)[:, 1]
    strat2_metrics = full_metrics(y_test, strat2_proba, "Strategy 2: SMOTE")
else:
    print("SMOTE skipped — imbalanced-learn not installed.")
    strat2_metrics = None
    strat2_proba = None

# Strategy 3: no class weighting, post-hoc threshold-moving
strat3_model = fit_lgb(X_train, y_train, class_weight_strategy="none")
strat3_proba = strat3_model.predict_proba(X_test)[:, 1]
strat3_metrics = full_metrics(y_test, strat3_proba, "Strategy 3: Threshold-moving")


In [ ]:
# Compare strategies
print("=== IMBALANCE STRATEGY COMPARISON ===")
print(f"{'Metric':<15} {'spw (curr)':>12} {'SMOTE':>12} {'No-weight':>12}")
for metric in ['roc_auc', 'pr_auc', 'brier', 'lift_5', 'lift_10', 'lift_20']:
    s1 = strat1_metrics[metric]
    s2 = strat2_metrics[metric] if strat2_metrics else float('nan')
    s3 = strat3_metrics[metric]
    print(f"{metric:<15} {s1:>12.4f} {s2:>12.4f} {s3:>12.4f}")

# Build lift curves comparison plot
fig, ax = plt.subplots(figsize=(9, 5))
for proba, name, color in [(strat1_proba, 'scale_pos_weight (current)', '#2E5C8A'),
                            (strat2_proba, 'SMOTE', '#D4A017') if strat2_proba is not None else (None, None, None),
                            (strat3_proba, 'No weighting (threshold-moving)', '#B22222')]:
    if proba is None: continue
    order = np.argsort(-proba)
    n = len(proba)
    cumulative_positives = np.cumsum(y_test[order])
    cumulative_share = np.arange(1, n+1) / n
    cumulative_recall = cumulative_positives / y_test.sum()
    ax.plot(cumulative_share*100, cumulative_recall*100, label=name, color=color, lw=2)
ax.plot([0, 100], [0, 100], color='gray', linestyle='--', alpha=0.5, label='Random baseline')
ax.set_xlabel('% of cohort contacted (sorted by predicted probability descending)')
ax.set_ylabel('% of actual churners captured')
ax.set_title('Cumulative gains curve — how many true churners are caught at each contact depth')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/p5_lift_curves.png", dpi=110, bbox_inches='tight')
plt.show()


## 7. Save all results

In [ ]:
results = {
    'baseline': base_metrics,
    'calibration': {
        'raw':      base_metrics,
        'platt':    platt_metrics,
        'isotonic': iso_metrics,
    },
    'imbalance': {
        'scale_pos_weight':  strat1_metrics,
        'smote':             strat2_metrics if strat2_metrics else None,
        'threshold_moving':  strat3_metrics,
    },
}
with open(f"{OUT_DIR}/p5_metrics.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved p5_metrics.json")

# Final summary card
print("\n" + "="*68)
print("  PHASE 5 EXPERIMENTS — SUMMARY")
print("="*68)
print(f"\nBaseline LightGBM:        PR-AUC={base_metrics['pr_auc']:.4f}  ROC-AUC={base_metrics['roc_auc']:.4f}  Brier={base_metrics['brier']:.5f}  Lift@10%={base_metrics['lift_10']:.2f}x")
print(f"\nCalibration:")
print(f"  Platt-scaled:           PR-AUC={platt_metrics['pr_auc']:.4f}  Brier={platt_metrics['brier']:.5f}")
print(f"  Isotonic-calibrated:    PR-AUC={iso_metrics['pr_auc']:.4f}  Brier={iso_metrics['brier']:.5f}")
print(f"\nImbalance strategies:")
print(f"  scale_pos_weight (curr): PR-AUC={strat1_metrics['pr_auc']:.4f}  Lift@10%={strat1_metrics['lift_10']:.2f}x")
if strat2_metrics:
    print(f"  SMOTE:                  PR-AUC={strat2_metrics['pr_auc']:.4f}  Lift@10%={strat2_metrics['lift_10']:.2f}x")
print(f"  Threshold-moving:       PR-AUC={strat3_metrics['pr_auc']:.4f}  Lift@10%={strat3_metrics['lift_10']:.2f}x")
print(f"\nError analysis at F1-optimal threshold ({best_thr:.4f}):")
print(f"  Overall precision: {tp/(tp+fp+1e-12):.3f}, recall: {tp/(tp+fn+1e-12):.3f}")
